# 01 - Data Exploration
สำรวจข้อมูลเบื้องต้นจากแหล่งข้อมูลต่างๆ

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

df = pd.read_csv(r'C:\Users\jarun\OneDrive\Desktop\pm25-early-warning-cnx\data\raw\firms_north2023-2025_viirs.csv')
df.head()

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
0,17.67002,97.82792,330.55,0.36,0.58,2023-01-01,606,N,VIIRS,n,2,294.15,3.00,D,0
1,17.60143,99.65153,306.65,0.41,0.45,2023-01-01,1837,N,VIIRS,n,2,286.79,0.60,N,0
2,19.91221,99.88639,304.38,0.42,0.45,2023-01-01,1836,N,VIIRS,n,2,288.19,0.31,N,0
3,19.32869,99.61866,307.10,0.43,0.46,2023-01-01,1836,N,VIIRS,n,2,286.21,0.59,N,0
4,18.77781,100.16030,300.27,0.39,0.44,2023-01-01,1836,N,VIIRS,n,2,285.76,0.58,N,0


In [22]:
df.shape

(330668, 15)

In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 330668 entries, 0 to 330667
Data columns (total 15 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   latitude    330668 non-null  float64
 1   longitude   330668 non-null  float64
 2   bright_ti4  330668 non-null  float64
 3   scan        330668 non-null  float64
 4   track       330668 non-null  float64
 5   acq_date    330668 non-null  str    
 6   acq_time    330668 non-null  int64  
 7   satellite   330668 non-null  str    
 8   instrument  330668 non-null  str    
 9   confidence  330668 non-null  str    
 10  version     330668 non-null  int64  
 11  bright_ti5  330668 non-null  float64
 12  frp         330668 non-null  float64
 13  daynight    330668 non-null  str    
 14  type        330668 non-null  int64  
dtypes: float64(7), int64(3), str(5)
memory usage: 43.5 MB


In [23]:
df['confidence'].value_counts()

confidence
n    282978
l     38392
h      9298
Name: count, dtype: int64

In [24]:
print(df.groupby('confidence')['bright_ti4'].agg(['mean', 'median']))
print(df.groupby('confidence')['frp'].agg(['mean', 'median']))

                  mean  median
confidence                    
h           367.000000  367.00
l           333.931705  333.64
n           324.594506  329.32
                 mean  median
confidence                   
h           39.434042   17.42
l           10.534379    3.96
n            5.874031    2.66


low confidence มี FRP median แค่ 4 MW ต่ำกว่า high เกือบ 5 เท่า และสังเกตว่า bright_ti4 (brightness temperature) ของ low confidence เฉลี่ยแค่ 333K ในขณะที่ high confidence อยู่ที่ 367K ทุก row เลย ถ้าเอา low เข้าไปด้วย จะทำให้ hotspot count วันนั้นพองขึ้นโดยที่ไม่ได้มีไฟจริงเพิ่ม โมเดลก็จะเรียนรู้ signal ที่ผิด

In [25]:
df1 = df[df['confidence'] != 'l']
df.shape

(330668, 15)

In [26]:
df['type'].value_counts()

type
0    330532
3       136
Name: count, dtype: int64

ในข้อมูลของคุณมี type=3 อยู่ 136 rows ดูพิกัดแล้วเช่น lat 18.84, lon 97.44 ซึ่งเป็นแนวชายแดนพม่าติดแม่น้ำ อาจเป็น reflection จากผิวน้ำ ไม่ใช่ไฟที่ส่งผลต่อ PM 2.5 ในจังหวัดเชียงใหม่ มีแค่ 136 rows จากทั้งหมด 330,668 ฟังดูน้อย แต่ถ้าไม่กรองออก จะมี noise ปนในวันที่มันปรากฏขึ้น

In [32]:
df2 = df1[df1['type'] != 3]
df2.shape

(292140, 15)

In [35]:
# ============================================================
# FIRMS Spatial Join — map lat/lon → จังหวัด + aggregate รายวัน
# ============================================================

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import requests, zipfile, io, os

# ── 1. โหลด Shapefile ขอบเขตจังหวัดไทย ───────────────────────
# ใช้ GADM Thailand level-1 (จังหวัด) — ฟรี ไม่ต้องสมัคร

SHP_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_THA_shp.zip"
SHP_DIR = "gadm_thailand"

if not os.path.exists(SHP_DIR):
    print("Downloading shapefile...")
    r = requests.get(SHP_URL)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall(SHP_DIR)
    print("Done.")

thailand = gpd.read_file(f"{SHP_DIR}/gadm41_THA_1.shp")

# เลือกเฉพาะ 8 จังหวัดภาคเหนือที่มีใน Open Meteo
TARGET_PROVINCES = [
    "Chiang Mai", "Chiang Rai", "Lampang", "Lamphun",
    "Mae Hong Son", "Nan", "Phayao", "Phrae"
]

# GADM ใช้ชื่อภาษาอังกฤษใน column NAME_1
# ตรวจดูชื่อจริงก่อน
print("GADM province names (sample):")
print(sorted(thailand["NAME_1"].unique())[:20])

# Map ชื่อ GADM → ชื่อที่ใช้ใน Open Meteo
NAME_MAP = {
    "Chiang Mai"  : "Chiang Mai",
    "Chiang Rai"  : "Chiang Rai",
    "Lampang"     : "Lampang",
    "Lamphun"     : "Lamphun",
    "Mae Hong Son": "Mae Hong Son",
    "Nan"         : "Nan",
    "Phayao"      : "Phayao",
    "Phrae"       : "Phrae",
}

north = thailand[thailand["NAME_1"].isin(NAME_MAP.keys())].copy()
north["Province"] = north["NAME_1"].map(NAME_MAP)
print(f"\nShapefile: {len(north)} provinces loaded")

# ── 2. โหลด FIRMS (ใช้ df2 ที่ตัด low confidence และ type=3 ออกแล้ว)
firms = df2.copy()
print(f"\nFIRMS rows: {len(firms):,}")

# ── 3. Spatial Join: lat/lon → Province ──────────────────────
print("\nRunning spatial join...")

# แปลง FIRMS เป็น GeoDataFrame (WGS84 = EPSG:4326)
geometry = [Point(xy) for xy in zip(firms["longitude"], firms["latitude"])]
firms_geo = gpd.GeoDataFrame(firms, geometry=geometry, crs="EPSG:4326")

# ทำให้ CRS ตรงกัน
north = north.to_crs("EPSG:4326")

# Point-in-polygon join
joined = gpd.sjoin(
    firms_geo,
    north[["Province", "geometry"]],
    how="left",
    predicate="within"
)

# จุดที่ตกนอกทุกจังหวัด (ชายแดน ฯลฯ) จะได้ Province = NaN
outside = joined["Province"].isna().sum()
print(f"Points outside all provinces: {outside:,} ({outside/len(joined)*100:.1f}%)")

# เก็บเฉพาะจุดที่อยู่ใน 8 จังหวัด
joined = joined.dropna(subset=["Province"])
print(f"Points matched to province: {len(joined):,}")

# ── 4. Aggregate รายวัน × จังหวัด ────────────────────────────
joined["acq_date"] = pd.to_datetime(joined["acq_date"])

hotspot_daily = (
    joined
    .groupby(["acq_date", "Province"])
    .agg(
        hotspot_count = ("frp", "count"),       # จำนวนจุดความร้อน
        frp_sum       = ("frp", "sum"),          # พลังงานไฟรวม (MW)
        frp_mean      = ("frp", "mean"),         # ความเข้มเฉลี่ย
    )
    .reset_index()
    .rename(columns={"acq_date": "date"})
)

print(f"\nHotspot daily shape: {hotspot_daily.shape}")
print(hotspot_daily.head(10))

# ── 5. ตรวจสอบ: วันที่ไม่มีไฟเลย → ต้อง fill 0 ──────────────
# สร้าง grid ครบทุกวัน × ทุกจังหวัด
all_dates = pd.date_range(
    start=joined["acq_date"].min(),
    end=joined["acq_date"].max(),
    freq="D"
)
full_grid = pd.MultiIndex.from_product(
    [all_dates, TARGET_PROVINCES],
    names=["date", "Province"]
).to_frame(index=False)

hotspot_daily = full_grid.merge(hotspot_daily, on=["date", "Province"], how="left")
hotspot_daily[["hotspot_count", "frp_sum", "frp_mean"]] = (
    hotspot_daily[["hotspot_count", "frp_sum", "frp_mean"]].fillna(0)
)

print(f"\nAfter filling zero-fire days: {hotspot_daily.shape}")
print(f"Date range: {hotspot_daily['date'].min()} → {hotspot_daily['date'].max()}")
print(f"Zero-fire rows: {(hotspot_daily['hotspot_count']==0).sum():,}")

# ── 6. Export ─────────────────────────────────────────────────
OUT = r"C:\Users\jarun\OneDrive\Desktop\pm25-early-warning-cnx\data\processed\firms_daily_by_province.csv"
hotspot_daily.to_csv(OUT, index=False)
print(f"\nSaved → {OUT}")
print(hotspot_daily.dtypes)

# ── ตัวอย่างผลลัพธ์ ───────────────────────────────────────────
print("\nตัวอย่าง 5 วันที่มีไฟเยอะสุด (Chiang Mai):")
cm = hotspot_daily[hotspot_daily["Province"] == "Chiang Mai"]
print(cm.nlargest(5, "hotspot_count")[["date", "hotspot_count", "frp_sum", "frp_mean"]])

Done.
GADM province names (sample):
['Amnat Charoen', 'Ang Thong', 'Bangkok Metropolis', 'Bueng Kan', 'Buri Ram', 'Chachoengsao', 'Chai Nat', 'Chaiyaphum', 'Chanthaburi', 'Chiang Mai', 'Chiang Rai', 'Chon Buri', 'Chumphon', 'Kalasin', 'Kamphaeng Phet', 'Kanchanaburi', 'Khon Kaen', 'Krabi', 'Lampang', 'Lamphun']

Shapefile: 8 provinces loaded

FIRMS rows: 292,140

Running spatial join...
Points outside all provinces: 153,115 (52.4%)
Points matched to province: 139,025

Hotspot daily shape: (3881, 5)
        date      Province  hotspot_count  frp_sum   frp_mean
0 2023-01-01    Chiang Rai              4    24.78   6.195000
1 2023-01-01       Lampang              2     2.67   1.335000
2 2023-01-01       Lamphun              1     8.05   8.050000
3 2023-01-01           Nan              1     6.77   6.770000
4 2023-01-01        Phayao              5    23.47   4.694000
5 2023-01-01         Phrae              2     3.63   1.815000
6 2023-01-02    Chiang Mai              1    12.44  12.440000


In [36]:
cm

,date,Province,hotspot_count,frp_sum,frp_mean
0,2023-01-01,Chiang Mai,0.0,0.00,0.000000
8,2023-01-02,Chiang Mai,1.0,12.44,12.440000
16,2023-01-03,Chiang Mai,5.0,26.77,5.354000
24,2023-01-04,Chiang Mai,5.0,20.66,4.132000
32,2023-01-05,Chiang Mai,2.0,7.74,3.870000
...,...,...,...,...,...
8728,2025-12-27,Chiang Mai,23.0,126.43,5.496957
8736,2025-12-28,Chiang Mai,31.0,97.68,3.150968
8744,2025-12-29,Chiang Mai,23.0,140.23,6.096957
8752,2025-12-30,Chiang Mai,9.0,46.51,5.167778
